# 05 — RAG Pipeline: Qualitative Demo on Clinical Queries

**Thesis reference:** Cap. 5, Sec. 5.4 — Módulo RAG  
**Objective:** Demonstrate the full Retrieval-Augmented Generation pipeline on 10 diverse clinical queries.

This notebook covers two ablation configurations:
- **A_base** — Base Llama 3.1 (8B) with no context (no retrieval, no fine-tuning)
- **C_base_rag** — Base Llama 3.1 (8B) + RAG retrieval from PubMed corpus

> **Note:** Configurations B (fine-tuned only) and D (fine-tuned + RAG) require Phase 3 (QLoRA fine-tuning), which is not yet complete. Those will be added in `06_evaluation.ipynb`.

The pipeline:
1. Encode the clinical query with **Bioformer-16L** (384d, mean pooling + L2 norm)
2. Retrieve **top-5** PubMed abstracts from Qdrant (`pubmed_abstracts`, cosine, min_year=2020)
3. Assemble a grounded prompt and call **Llama 3.1 (8B)** via HF Inference API
4. Display retrieved documents + LLM response with `[PMID: XXXXX]` citations

---

### Prerequisites
- `docker compose up -d` (Qdrant at localhost:6333, MLflow at localhost:5000)
- `uv sync --extra dev`
- `.env` configured (`HF_TOKEN`, `NCBI_EMAIL`, `QDRANT_HOST`)
- `data/raw/pubmed_bulk_corpus.json` present (~29K abstracts from notebook 02)

In [66]:
# Prerequisites check — validate all runtime dependencies before loading any model
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(override=True)

print(f"Python {sys.version}")

# Qdrant
from qdrant_client import QdrantClient as _QC
_qc = _QC(host=os.getenv("QDRANT_HOST", "localhost"), port=int(os.getenv("QDRANT_PORT", 6333)))
print(f"Qdrant connected: {_qc.get_collections()}")

# MLflow
import mlflow
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

# Corpus
CORPUS_PATH = Path("../data/raw/pubmed_bulk_corpus.json")
assert CORPUS_PATH.exists(), f"Corpus not found: {CORPUS_PATH}"
print(f"Corpus file: {CORPUS_PATH} ({CORPUS_PATH.stat().st_size / 1e6:.1f} MB)")

# HF token
assert os.getenv("HF_TOKEN"), "HF_TOKEN not set — required for Llama 3.1 via HF Inference API"
print("HF_TOKEN configured")

# GPU (informational only — generator uses HF API, embedder runs locally)
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU detected — embedding will run on CPU (expect ~5 min for full corpus indexing)")

Python 3.12.12 (main, Feb 12 2026, 00:42:34) [Clang 21.1.4 ]
Qdrant connected: collections=[CollectionDescription(name='pubmed_abstracts')]
MLflow tracking URI: http://localhost:5001
Corpus file: ../data/raw/pubmed_bulk_corpus.json (80.0 MB)
HF_TOKEN configured
No GPU detected — embedding will run on CPU (expect ~5 min for full corpus indexing)


In [67]:
# Load RAG configuration from configs/rag.yaml
sys.path.insert(0, "..")
from src.config import load_config

cfg = load_config("rag")

print("RAG configuration loaded:")
print(f"  embedding.model        : {cfg['embedding']['model']}")
print(f"  embedding.batch_size   : {cfg['embedding']['batch_size']}")
print(f"  vector_store.collection: {cfg['vector_store']['collection_name']}")
print(f"  vector_store.host:port : {cfg['vector_store']['host']}:{cfg['vector_store']['port']}")
print(f"  retrieval.top_k        : {cfg['retrieval']['top_k']}")
print(f"  retrieval.min_year     : {cfg['retrieval']['metadata_filters']['min_year']}")
print(f"  generation.temperature : {cfg['generation']['temperature']}")
print(f"  generation.max_tokens  : {cfg['generation']['max_tokens']}")

RAG configuration loaded:
  embedding.model        : bioformers/bioformer-16L
  embedding.batch_size   : 32
  vector_store.collection: pubmed_abstracts
  vector_store.host:port : localhost:6333
  retrieval.top_k        : 10
  retrieval.min_year     : 2015
  generation.temperature : 0.1
  generation.max_tokens  : 1024


## Step 1: Index PubMed Corpus in Qdrant

The indexing cell is **idempotent**: it only runs if the `pubmed_abstracts` collection does not yet exist or has the wrong vector dimension (e.g. a stale 768d PubMedBERT collection).

When indexing runs:
1. Load `pubmed_bulk_corpus.json` (~29K abstracts)
2. Encode with **Bioformer-16L** (explicit mean pooling, L2 normalization → 384d vectors)
3. Upsert in batches of 256 into Qdrant with cosine distance

The `year` field is stored as an integer (defaulting to 0 for missing values) because Qdrant's range filter requires numeric comparison.

In [68]:
import json
import numpy as np
from tqdm.auto import tqdm
import transformers
from sentence_transformers import SentenceTransformer, models
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

transformers.logging.set_verbosity_error()

COLLECTION_NAME = cfg["vector_store"]["collection_name"]
EXPECTED_DIM = 384

qdrant = QdrantClient(
    host=cfg["vector_store"]["host"],
    port=cfg["vector_store"]["port"],
)

SKIP_INDEXING = False
if qdrant.collection_exists(COLLECTION_NAME):
    info = qdrant.get_collection(COLLECTION_NAME)
    actual_dim = info.config.params.vectors.size
    point_count = info.points_count
    if actual_dim == EXPECTED_DIM:
        print(f"Collection '{COLLECTION_NAME}' already exists: {point_count} points, dim={actual_dim}. Skipping indexing.")
        SKIP_INDEXING = True
    else:
        print(f"Dimension mismatch: found {actual_dim}d, expected {EXPECTED_DIM}d. Recreating collection.")
        qdrant.delete_collection(COLLECTION_NAME)

if not SKIP_INDEXING:
    print("Loading corpus...")
    with open(CORPUS_PATH) as f:
        corpus = json.load(f)
    print(f"Loaded {len(corpus)} abstracts")

    print(f"Loading embedding model: {cfg['embedding']['model']}")
    word_model = models.Transformer(cfg["embedding"]["model"], max_seq_length=512)
    pooling_model = models.Pooling(
        word_model.get_word_embedding_dimension(),
        pooling_mode_mean_tokens=True,
        pooling_mode_cls_token=False,
        pooling_mode_max_tokens=False,
    )
    st_model = SentenceTransformer(modules=[word_model, pooling_model])
    print(f"Model loaded | dim={st_model.get_sentence_embedding_dimension()}")

    texts = [p.get("abstract", "") for p in corpus]
    print("Encoding abstracts...")
    embeddings = st_model.encode(
        texts,
        batch_size=cfg["embedding"]["batch_size"],
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    print(f"Encoded {len(embeddings)} vectors of dim {embeddings.shape[1]}")

    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EXPECTED_DIM, distance=Distance.COSINE),
    )
    print(f"Collection '{COLLECTION_NAME}' created")

    UPSERT_BATCH = 256
    for batch_start in tqdm(range(0, len(corpus), UPSERT_BATCH), desc="Upserting"):
        batch_docs = corpus[batch_start : batch_start + UPSERT_BATCH]
        batch_emb = embeddings[batch_start : batch_start + UPSERT_BATCH]
        points = [
            PointStruct(
                id=batch_start + j,
                vector=batch_emb[j].tolist(),
                payload={
                    "pmid": batch_docs[j].get("pmid", ""),
                    "title": batch_docs[j].get("title", ""),
                    "abstract": batch_docs[j].get("abstract", ""),
                    "mesh_category": batch_docs[j].get("mesh_category", ""),
                    "year": int(batch_docs[j].get("year") or 0),
                    "journal": batch_docs[j].get("journal", ""),
                    "authors": batch_docs[j].get("authors", ""),
                },
            )
            for j in range(len(batch_docs))
        ]
        qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

    final_count = qdrant.get_collection(COLLECTION_NAME).points_count
    print(f"Indexed {final_count} points into '{COLLECTION_NAME}'")

Collection 'pubmed_abstracts' already exists: 29108 points, dim=384. Skipping indexing.


## Step 2: Define RAG Components

The four classes below implement the full RAG pipeline. They are defined inline here as working prototype code; the same implementations will be extracted to `src/rag/` in a later phase for use by the Gradio interface.

- **`Embedder`** — wraps Bioformer-16L with explicit mean pooling and L2 normalization
- **`Retriever`** — queries Qdrant with a year filter and returns ranked document dicts
- **`Generator`** — assembles the grounded prompt and calls Llama 3.1 via HF Inference API
- **`RAGPipeline`** — orchestrates the three components into a single `run(query)` call

In [69]:
# Embedder — Bioformer-16L with explicit mean pooling

class Embedder:
    def __init__(self, model_name: str, max_length: int = 512):
        self.model_name = model_name
        self.max_length = max_length
        self._model: SentenceTransformer | None = None

    def load_model(self) -> None:
        word_model = models.Transformer(self.model_name, max_seq_length=self.max_length)
        pooling_model = models.Pooling(
            word_model.get_word_embedding_dimension(),
            pooling_mode_mean_tokens=True,
            pooling_mode_cls_token=False,
            pooling_mode_max_tokens=False,
        )
        self._model = SentenceTransformer(modules=[word_model, pooling_model])

    def encode(self, texts: list[str], batch_size: int = 32) -> np.ndarray:
        if self._model is None:
            self.load_model()
        return self._model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        ).astype(np.float32)

    @property
    def embedding_dim(self) -> int:
        if self._model is None:
            self.load_model()
        return self._model.get_sentence_embedding_dimension()


embedder = Embedder(cfg["embedding"]["model"])
embedder.load_model()
print(f"Embedder ready: {cfg['embedding']['model']} | dim={embedder.embedding_dim}")

Loading weights:   0%|          | 0/263 [00:00<?, ?it/s]

Embedder ready: bioformers/bioformer-16L | dim=384


In [70]:
# Retriever — Qdrant semantic search with year metadata filter
from qdrant_client.models import Filter, FieldCondition, Range

class Retriever:
    def __init__(self, embedder: Embedder, client: QdrantClient, collection_name: str):
        self.embedder = embedder
        self.client = client
        self.collection_name = collection_name

    def search(
        self,
        query_text: str,
        top_k: int = 5,
        min_year: int | None = 2020,
    ) -> list[dict]:
        query_vec = self.embedder.encode([query_text])[0].tolist()

        year_filter = (
            Filter(must=[FieldCondition(key="year", range=Range(gte=min_year))])
            if min_year is not None
            else None
        )

        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_vec,
            query_filter=year_filter,
            limit=top_k,
        )

        return [
            {
                "rank": i + 1,
                "pmid": hit.payload.get("pmid", ""),
                "title": hit.payload.get("title", ""),
                "abstract": hit.payload.get("abstract", ""),
                "mesh_category": hit.payload.get("mesh_category", ""),
                "year": hit.payload.get("year", ""),
                "journal": hit.payload.get("journal", ""),
                "score": round(hit.score, 4),
            }
            for i, hit in enumerate(results.points)
        ]


retriever = Retriever(
    embedder=embedder,
    client=qdrant,
    collection_name=COLLECTION_NAME,
)
print("Retriever ready")

Retriever ready


In [71]:
# Generator — Llama 3.1 (8B) via HF Inference API
from huggingface_hub import InferenceClient

class Generator:
    def __init__(
        self,
        model_id: str,
        system_prompt: str,
        max_tokens: int = 1024,
        temperature: float = 0.1,
        hf_token: str | None = None,
    ):
        self.model_id = model_id
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.temperature = temperature
        self._client = InferenceClient(token=hf_token)

    def build_prompt(self, query: str, docs: list[dict]) -> str:
        context_parts = []
        for doc in docs:
            context_parts.append(
                f"[{doc['rank']}] PMID: {doc['pmid']} | Year: {doc['year']} | MeSH: {doc['mesh_category']}\n"
                f"Title: {doc['title']}\n"
                f"Abstract: {doc['abstract']}"
            )
        context = "\n\n---\n\n".join(context_parts)
        return (
            f"=== RETRIEVED EVIDENCE ===\n\n{context}\n\n"
            f"=== END OF EVIDENCE ===\n\n"
            f"CLINICAL QUERY: {query}\n\n"
            f"EVIDENCE-BASED RESPONSE:"
        )

    def generate(self, prompt: str) -> str:
        response = self._client.chat_completion(
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt},
            ],
            model=self.model_id,
            max_tokens=self.max_tokens,
            temperature=self.temperature,
        )
        return response.choices[0].message.content

    def generate_base(self, query: str) -> str:
        """Config A_base: no retrieved context."""
        bare_prompt = f"CLINICAL QUERY: {query}\n\nEVIDENCE-BASED RESPONSE:"
        return self.generate(bare_prompt)


BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

generator = Generator(
    model_id=BASE_MODEL,
    system_prompt=cfg["generation"]["system_prompt"],
    max_tokens=cfg["generation"]["max_tokens"],
    temperature=cfg["generation"]["temperature"],
    hf_token=os.getenv("HF_TOKEN"),
)
print(f"Generator ready: {generator.model_id}")

Generator ready: meta-llama/Llama-3.1-8B-Instruct


In [72]:
# RAGPipeline — orchestrates Retriever + Generator

class RAGPipeline:
    def __init__(
        self,
        retriever: Retriever,
        generator: Generator,
        top_k: int = 5,
        min_year: int = 2020,
    ):
        self.retriever = retriever
        self.generator = generator
        self.top_k = top_k
        self.min_year = min_year

    def run(self, query: str) -> dict:
        docs = self.retriever.search(query, top_k=self.top_k, min_year=self.min_year)
        prompt = self.generator.build_prompt(query, docs)
        response = self.generator.generate(prompt)
        avg_score = round(sum(d["score"] for d in docs) / len(docs), 4) if docs else 0.0
        return {
            "query": query,
            "docs": docs,
            "prompt": prompt,
            "response": response,
            "avg_score": avg_score,
        }


pipeline = RAGPipeline(
    retriever=retriever,
    generator=generator,
    top_k=cfg["retrieval"]["top_k"],
    min_year=cfg["retrieval"]["metadata_filters"]["min_year"],
)
print(f"RAGPipeline assembled | top_k={pipeline.top_k} | min_year={pipeline.min_year}")

RAGPipeline assembled | top_k=10 | min_year=2015


## Step 3: 9 Clinical Queries

| ID | MeSH Category | Clinical Topic |
|----|--------------|----------------|
| Q01 | Cardiovascular Diseases | Anticoagulation in atrial fibrillation + CKD |
| Q02 | Nervous System Diseases | DMTs for relapsing-remitting MS |
| Q03 | Mental Disorders | Treatment-resistant major depression |
| Q04 | Endocrine System Diseases | Glycemic targets in elderly T2DM with CVD risk |
| Q05 | Musculoskeletal Diseases | bDMARDs in RA after methotrexate failure |
| Q06 | Respiratory Tract Diseases | Severe persistent asthma — inhaler escalation |
| Q07 | Digestive System Diseases | H. pylori eradication with clarithromycin resistance |
| Q08 | Bacterial Infections and Mycoses | Carbapenem-resistant Klebsiella in ICU |
| Q09 | Urogenital Diseases | PSA screening harms/benefits in men 55-69 |

In [75]:
CLINICAL_QUERIES = [
    {
        "id": "Q01",
        "mesh_category": "Cardiovascular Diseases",
        "query": "What are the current evidence-based recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation and chronic kidney disease?",
    },
    {
        "id": "Q02",
        "mesh_category": "Nervous System Diseases",
        "query": "What disease-modifying therapies are recommended for relapsing-remitting multiple sclerosis in patients with high disease activity?",
    },
    {
        "id": "Q03",
        "mesh_category": "Mental Disorders",
        "query": "What pharmacological and non-pharmacological interventions are most effective for treatment-resistant major depressive disorder?",
    },
    {
        "id": "Q04",
        "mesh_category": "Endocrine System Diseases",
        "query": "What are the recommended glycemic targets and preferred antidiabetic agents for elderly patients with type 2 diabetes and high cardiovascular risk?",
    },
    {
        "id": "Q05",
        "mesh_category": "Musculoskeletal Diseases",
        "query": "What is the evidence for biological disease-modifying antirheumatic drugs in patients with rheumatoid arthritis who have failed methotrexate therapy?",
    },
    {
        "id": "Q06",
        "mesh_category": "Respiratory Tract Diseases",
        "query": "What are the recommended maintenance inhaler regimens for patients with severe persistent asthma uncontrolled on medium-dose inhaled corticosteroids?",
    },
    {
        "id": "Q07",
        "mesh_category": "Digestive System Diseases",
        "query": "What is the optimal therapeutic strategy for Helicobacter pylori eradication in regions with high clarithromycin resistance rates?",
    },
    {
        "id": "Q08",
        "mesh_category": "Bacterial Infections and Mycoses",
        "query": "What antimicrobial regimens are recommended for carbapenem-resistant Klebsiella pneumoniae infections in critically ill patients?",
    },
    {
        "id": "Q09",
        "mesh_category": "Urogenital Diseases",
        "query": "What is the current evidence for prostate-specific antigen screening in men aged 55-69 and what are the associated harms and benefits?",
    },
]

print(f"{len(CLINICAL_QUERIES)} queries across {len(set(q['mesh_category'] for q in CLINICAL_QUERIES))} MeSH categories")

9 queries across 9 MeSH categories


In [ ]:
# Run all 9 queries through the RAG pipeline
import time
import pandas as pd

all_results = []

for q in CLINICAL_QUERIES:
    print(f"\n{'='*72}")
    print(f"[{q['id']}] {q['mesh_category']}")
    print(f"Query: {q['query']}")

    t0 = time.perf_counter()
    result = pipeline.run(q["query"])
    elapsed = round(time.perf_counter() - t0, 2)

    result["query_id"] = q["id"]
    result["mesh_category"] = q["mesh_category"]
    result["elapsed_s"] = elapsed
    all_results.append(result)

    docs_df = pd.DataFrame(
        [
            {
                "Rank": d["rank"],
                "PMID": d["pmid"],
                "Year": d["year"],
                "MeSH Category": d["mesh_category"],
                "Score": d["score"],
                "Title": d["title"][:80] + ("..." if len(d["title"]) > 80 else ""),
            }
            for d in result["docs"]
        ]
    )
    display(docs_df)

    print(f"\nLLM Response (avg_score={result['avg_score']:.4f}, {elapsed}s):")
    print(result["response"])


[Q01] Cardiovascular Diseases
Query: What are the current evidence-based recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation and chronic kidney disease?


,Rank,PMID,Year,MeSH Category,Score,Title
0,1,38823454,2024,Digestive System Diseases,0.8332,Anticoagulation for stroke prevention in atria...
1,2,31119266,2019,Nervous System Diseases,0.8286,Oral anticoagulation in patients with non-valv...
2,3,30442731,2018,Neoplasms,0.8280,NCCN Guidelines Insights: Cancer-Associated Ve...
3,4,38845253,2024,Virus Diseases,0.8265,EASL position paper on clinical follow-up afte...
4,5,27126598,2016,Nervous System Diseases,0.8255,"North American Thrombosis Forum, AF Action Ini..."
5,6,31441247,2019,Nutritional and Metabolic Diseases,0.8229,2019 Clinical Practice Guidelines for Type 2 D...
6,7,30571525,2018,Cardiovascular Diseases,0.8224,Antithrombotic Therapy in Patients With Atrial...
7,8,30877397,2019,Cardiovascular Diseases,0.8216,2018 Cholesterol Guidelines: Key Topics in Pri...
8,9,25532421,2015,Cardiovascular Diseases,0.8213,The 2014 Canadian Cardiovascular Society Heart...
9,10,35300861,2022,Cardiovascular Diseases,0.8211,EASL Clinical Practice Guidelines on preventio...



LLM Response (avg_score=0.8251, 1.02s):
Based on the provided evidence, there is limited information on the specific recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation and chronic kidney disease. However, we can make some general inferences and recommendations based on the available literature.

The CHA2DS2-VASc score is a widely used tool to assess the risk of stroke in patients with atrial fibrillation [PMID: 31119266]. A score of 1 is considered low risk, and the decision to anticoagulate in these patients is often challenging due to the balance between the risk of stroke and bleeding [PMID: 31119266].

For patients with chronic kidney disease (CKD), the risk of bleeding and thrombosis is increased, and anticoagulation therapy should be individualized based on the patient's specific risk factors and clinical context [PMID: 35300861].

The European Society of Cardiology recommends the use of direct oral anticoagulants (DOACs) in patients wi

,Rank,PMID,Year,MeSH Category,Score,Title
0,1,26237913,2015,Nervous System Diseases,0.8580,[Update on Current Care Guideline: Multiple sc...
1,2,28254463,2017,Digestive System Diseases,0.8277,Use of corticosteroids and immunosuppressive d...
2,3,39513243,2024,Musculoskeletal Diseases,0.8258,South African Rheumatism and Arthritis Associa...
3,4,39508216,2024,Musculoskeletal Diseases,0.8112,South African Rheumatism and Arthritis Associa...
4,5,29871738,2018,Bacterial Infections and Mycoses,0.8030,Prevention and treatment of tuberculosis infec...
5,6,32552502,2020,Digestive System Diseases,0.8026,European Guideline on IgG4-related digestive d...
6,7,39513242,2024,Musculoskeletal Diseases,0.8002,South African Rheumatism and Arthritis Associa...
7,8,39737584,2024,Nervous System Diseases,0.7963,Disease-modifying therapy in multiple sclerosi...
8,9,37087905,2023,Respiratory Tract Diseases,0.7933,Guidelines for the management of chronic cough...
9,10,30926722,2019,Skin and Connective Tissue Diseases,0.7923,2019 update of the EULAR recommendations for t...



LLM Response (avg_score=0.8110, 0.72s):
Based on the provided evidence, for patients with relapsing-remitting multiple sclerosis (RRMS) and high disease activity, the following disease-modifying therapies (DMTs) are recommended:

1. Alemtuzumab [PMID: 26237913]
2. Fingolimod [PMID: 26237913]
3. Natalizumab [PMID: 26237913]

These therapies are considered as second-line treatments for RRMS with active disease during first-line immunomodulating drug (IMD) treatment. It is essential to note that the choice of DMT should be individualized and based on the patient's specific needs, comorbidities, and disease characteristics.

Additionally, recent guidelines from the Polish Neurological Society [PMID: 39737584] recommend considering the following factors when choosing a DMT for RRMS:

* Disease progression and prognosis
* Comorbidities
* Administration of other drugs during DMT
* Age of the patient
* Potential adverse effects of the treatment

It is crucial for healthcare professionals to c

...


LLM Response (avg_score=0.8054, 0.74s):
Based on the provided evidence, the optimal therapeutic strategy for Helicobacter pylori eradication in regions with high clarithromycin resistance rates is to use alternative therapies, including new molecular methods, and antibiotic stewardship.

According to the Italian Society of Gastroenterology and the Italian Society of Digestive Endoscopy guidelines [PMID: 35831212], the increasing resistance to antibiotics, including macrolides and fluoroquinolones, is a major challenge in the management of H. pylori. The guidelines recommend the use of alternative therapies, including new molecular methods, to improve eradication rates.

Similarly, the First regional consensus on the management of Helicobacter pylori infection in the Middle East [PMID: 40741103] emphasizes the need for routine antibiotic susceptibility testing at the national level, the use of alternative therapies, and antibiotic stewardship to address the challenge of clarithromycin 

,Rank,PMID,Year,MeSH Category,Score,Title
0,1,35613961,2022,Bacterial Infections and Mycoses,0.8227,AAUS guideline for acute uncomplicated pyelone...
1,2,41130038,2025,Bacterial Infections and Mycoses,0.8075,Guideline for antimicrobial treatment of multi...
2,3,25976750,2015,Urogenital Diseases,0.8022,Executive summary. Management of urinary tract...
3,4,40967383,2025,Bacterial Infections and Mycoses,0.8002,Management of infections caused by Extended-sp...
4,5,28832488,2017,Musculoskeletal Diseases,0.7929,"Committee Opinion No. 717: Sulfonamides, Nitro..."
5,6,28832479,2017,Musculoskeletal Diseases,0.7929,Committee Opinion No. 717 Summary: Sulfonamide...
6,7,25563393,2015,Bacterial Infections and Mycoses,0.7924,Executive summary of the diagnosis and antimic...
7,8,29290231,2017,Bacterial Infections and Mycoses,0.7917,Antimicrobial treatment of diarrhea/acute gast...
8,9,28267082,2017,Bacterial Infections and Mycoses,0.7907,Guidelines for the Prophylaxis of Pneumocystis...
9,10,40793894,2025,Bacterial Infections and Mycoses,0.7906,[Consensus on the diagnosis and treatment of c...



LLM Response (avg_score=0.7984, 0.53s):
Based on the provided evidence, for carbapenem-resistant Klebsiella pneumoniae infections in critically ill patients, the following antimicrobial regimens are recommended:

1. Novel B-lactam/B-lactamase inhibitors (BL/BLI) as initial empirical therapy in patients with septic shock and certain epidemiological conditions, such as central venous catheters or endovascular devices-associated bacteremias, endocarditis, mediastinitis, intra-abdominal infections, and post-neurosurgical infections [PMID: 40793894].
2. BL/BLI as directed therapy for confirmed microbiological CRE infections for:
   - Septic shock
   - Bacteremia/infections with any focus and INCREMENT score = 8
   - Bacteremia in severely immunocompromised or neonatal patients
   - Endocarditis/non-removed endovascular focus (implant)
   - Mediastinitis
   - Post-neurosurgical CNS infection [PMID: 40793894].

Specifically, the recommended BL/BLI combinations include cefepime-enmetazobactam

,Rank,PMID,Year,MeSH Category,Score,Title
0,1,25419853,2015,Nutritional and Metabolic Diseases,0.8176,Screening for vitamin D deficiency in adults: ...
1,2,39041471,2024,Neoplasms,0.8081,The South African Prostate Cancer Screening Gu...
2,3,25798805,2015,Endocrine System Diseases,0.7996,Screening for thyroid dysfunction: U.S. Preven...
3,4,29801017,2018,Neoplasms,0.7991,Screening for Prostate Cancer: US Preventive S...
4,5,27304597,2016,Neoplasms,0.7989,Screening for Colorectal Cancer: US Preventive...
5,6,36031558,2022,Neoplasms,0.7968,Discordant Endorsement of Prostate Cancer Biom...
6,7,31536851,2019,"Congenital, Hereditary, and Neonatal Diseases ...",0.7967,Current management of children and young peopl...
7,8,29450531,2018,Endocrine System Diseases,0.7954,Screening for Ovarian Cancer: US Preventive Se...
8,9,27532917,2016,Nutritional and Metabolic Diseases,0.7948,Screening for Lipid Disorders in Children and ...
9,10,35076659,2022,Cardiovascular Diseases,0.7915,Screening for Atrial Fibrillation: US Preventi...



LLM Response (avg_score=0.7998, 0.56s):
Based on the provided evidence, the current recommendation for prostate-specific antigen (PSA) screening in men aged 55-69 is as follows:

The US Preventive Services Task Force (USPSTF) recommends that the decision to undergo periodic PSA-based screening for prostate cancer should be an individual one and should include discussion of the potential benefits and harms of screening with their clinician. Screening offers a small potential benefit of reducing the chance of death from prostate cancer in some men, but many men will experience potential harms of screening, including false-positive results that require additional testing and possible prostate biopsy; overdiagnosis and overtreatment; and treatment complications, such as incontinence and erectile dysfunction.

The USPSTF concludes with moderate certainty that the net benefit of PSA-based screening for prostate cancer in men aged 55 to 69 years is small for some men. The decision to screen 

## Step 4: Base Model vs RAG-Augmented Comparison

We compare **config A_base** (no context) vs **config C_base_rag** (top-k retrieved PubMed abstracts) on the queries:

In [ ]:
base_responses = {}

for q in CLINICAL_QUERIES:
    qid = q["id"]
    print(f"Generating base response for [{qid}]...")
    base_responses[qid] = generator.generate_base(q["query"])
    print(f"  Done ({len(base_responses[qid])} chars)")

Generating base response for [Q01]...
  Done (3554 chars)
Generating base response for [Q02]...
  Done (3694 chars)
Generating base response for [Q03]...
  Done (3971 chars)
Generating base response for [Q04]...
  Done (2246 chars)
Generating base response for [Q05]...
  Done (3025 chars)
Generating base response for [Q06]...
  Done (2082 chars)
Generating base response for [Q07]...
  Done (3375 chars)
Generating base response for [Q08]...
  Done (3055 chars)
Generating base response for [Q09]...
  Done (4143 chars)


In [78]:
# Side-by-side comparison display
for q in CLINICAL_QUERIES:
    qid = q["id"]
    rag_result = next(r for r in all_results if r["query_id"] == qid)
    query_text = q["query"]

    print(f"\n{'='*72}")
    print(f"[{qid}] {rag_result['mesh_category']}")
    print(f"Query: {query_text}")

    print(f"\n{'─'*34} A_base (no context) {'─'*18}")
    print(base_responses[qid])

    print(f"\n{'─'*34} C_base_rag (top-{pipeline.top_k}, min_year={pipeline.min_year}) {'─'*6}")
    print(f"Retrieved docs avg_score={rag_result['avg_score']:.4f}:")
    for d in rag_result["docs"]:
        print(f"  [{d['rank']}] PMID:{d['pmid']} ({d['year']}) score={d['score']:.4f} — {d['title'][:70]}")
    print()
    print(rag_result["response"])


[Q01] Cardiovascular Diseases
Query: What are the current evidence-based recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation and chronic kidney disease?

────────────────────────────────── A_base (no context) ──────────────────
Based on recent medical literature, the current evidence-based recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation (NVAF) and chronic kidney disease (CKD) are as follows:

1. **Oral Anticoagulation Therapy**: Patients with NVAF and CKD should receive oral anticoagulation therapy to reduce the risk of stroke and systemic embolism [1, 2]. The choice of anticoagulant should be individualized based on the patient's renal function, bleeding risk, and other comorbidities.

2. **Renal Function-Based Guidelines**: The American College of Cardiology (ACC) and the American Heart Association (AHA) recommend the following anticoagulation strategies based on renal function [3]:
	* For patien

In [80]:
# Log experiment to MLflow
mlflow.set_experiment("05-rag-pipeline-demo")

with mlflow.start_run(run_name="base_rag_qualitative_demo_v1") as run:
    mlflow.log_param("embedding_model", cfg["embedding"]["model"])
    mlflow.log_param("llm_model", generator.model_id)
    mlflow.log_param("collection_name", COLLECTION_NAME)
    mlflow.log_param("top_k", pipeline.top_k)
    mlflow.log_param("min_year", pipeline.min_year)
    mlflow.log_param("temperature", generator.temperature)
    mlflow.log_param("max_tokens", generator.max_tokens)
    mlflow.log_param("n_queries", len(all_results))
    mlflow.log_param("fine_tuned", False)
    mlflow.log_param("ablation_config", "C_base_rag")

    for r in all_results:
        mlflow.log_metric(f"avg_score_{r['query_id']}", r["avg_score"])
        if r["docs"]:
            mlflow.log_metric(f"top1_score_{r['query_id']}", r["docs"][0]["score"])

    mlflow.log_metric(
        "overall_avg_retrieval_score",
        sum(r["avg_score"] for r in all_results) / len(all_results),
    )

    results_artifact = [
        {
            "query_id": r["query_id"],
            "mesh_category": r["mesh_category"],
            "query": r["query"],
            "avg_score": r["avg_score"],
            "docs": r["docs"],
            "response": r["response"],
            "elapsed_s": r["elapsed_s"],
        }
        for r in all_results
    ]
    mlflow.log_dict(results_artifact, "rag_demo_results.json")

    comparison_artifact = {
        qid: {
            "query": next(q["query"] for q in CLINICAL_QUERIES if q["id"] == qid),
            "base_response": base_responses[qid],
            "rag_response": next(r["response"] for r in all_results if r["query_id"] == qid),
            "rag_avg_score": next(r["avg_score"] for r in all_results if r["query_id"] == qid),
        }
        for qid in BASE_COMPARISON_IDS
    }
    mlflow.log_dict(comparison_artifact, "base_vs_rag_comparison.json")

    print(f"Run logged: {run.info.run_id}")
    print(f"MLflow dashboard: {mlflow.get_tracking_uri()}")

Run logged: 8655b4a3e9094ada89f4de099f93bf68
MLflow dashboard: http://localhost:5001
🏃 View run base_rag_qualitative_demo_v1 at: http://localhost:5001/#/experiments/1/runs/8655b4a3e9094ada89f4de099f93bf68
🧪 View experiment at: http://localhost:5001/#/experiments/1
